# Week 09 Lab — Retrieval-Augmented Generation (RAG) and Large Language Models

## Learning Goals
- Understand the motivation for **Retrieval-Augmented Generation (RAG)** and how it extends LLMs.
- Build a **document chunking and embedding** pipeline from scratch.
- Implement a **cosine similarity vector store** and nearest-neighbour retrieval.
- Construct a full **RAG pipeline**: retrieve → augment → generate.
- Evaluate retrieval quality with **precision@k** and **MRR**.
- Analyse **hallucination** and the effect of context quality on generation.

## Part 0 — Setup

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers torch numpy pandas matplotlib scikit-learn

In [ ]:
import re
import math
import json
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

random.seed(42)
np.random.seed(42)
print("All imports OK.")

---
## Part 1 — LLMs and the Knowledge Problem

Large Language Models are pre-trained on massive corpora and encode a great deal of world knowledge in their parameters. However, they have two fundamental limitations:

1. **Knowledge cutoff** — the model has no information about events after its training data was collected.
2. **Hallucination** — the model may generate plausible-sounding but factually incorrect text.

**RAG** addresses both by supplying relevant external documents as context at inference time: *retrieve → augment prompt → generate*.

### Task 1A — Identifying Hallucination-Prone Queries

For each query below, classify it as **Low Risk** (LLM likely knows the answer) or **High Risk** (LLM may hallucinate) and explain why.

| # | Query | Risk Level | Reason |
|---|-------|------------|--------|
| 1 | What is the capital of France? | | |
| 2 | What did the CEO of Acme Corp say in their Q3 2024 earnings call? | | |
| 3 | Explain the concept of backpropagation in neural networks. | | |
| 4 | What are the current visa requirements for Vietnamese citizens travelling to Japan? | | |
| 5 | What is 17 multiplied by 34? | | |
| 6 | Summarise the key findings from the Smith et al. 2024 Nature paper on protein folding. | | |
| 7 | What are the main themes in Shakespeare's Hamlet? | | |
| 8 | What is the current stock price of Apple? | | |

_Fill in the table above._

### Questions
1. What is the difference between **parametric knowledge** (stored in model weights) and **non-parametric knowledge** (retrieved from an external source)?
2. Name two scenarios where RAG would be preferred over a fine-tuned LLM, and two scenarios where fine-tuning would be preferred over RAG.
3. What is **context stuffing** and why is it not a scalable solution to the knowledge problem?

_Your answers here._

---
## Part 2 — Document Chunking

RAG systems must first split source documents into smaller **chunks** before embedding them. Chunk size affects both retrieval precision and the amount of context available to the generator.

### Chunking strategies:
- **Fixed-size**: split every N characters or tokens, with optional overlap.
- **Sentence-boundary**: split at sentence ends to preserve semantic units.
- **Paragraph-boundary**: split at double-newlines.
- **Recursive character**: try paragraph → sentence → word boundaries in order.

In [ ]:
# Sample knowledge base — five short passages about NLP topics
DOCUMENTS = [
    {
        "id": "doc_001",
        "title": "Transformer Architecture",
        "text": """The Transformer architecture, introduced by Vaswani et al. in 2017, 
revolutionised natural language processing. Unlike recurrent neural networks, 
Transformers process all tokens in parallel using a self-attention mechanism. 
The core idea is the scaled dot-product attention: given query Q, key K, and 
value V matrices, attention is computed as softmax(QK^T / sqrt(d_k)) * V. 
The multi-head variant applies attention h times with different learned projections, 
allowing the model to attend to information from different representation subspaces. 
Transformers also use positional encodings to inject sequence order information, 
since self-attention is inherently permutation-invariant. The original paper used 
sinusoidal positional encodings, though learned positional embeddings are now common."""
    },
    {
        "id": "doc_002",
        "title": "BERT Pre-training",
        "text": """BERT (Bidirectional Encoder Representations from Transformers), introduced by 
Devlin et al. in 2019, is a landmark pre-trained language model. BERT is trained 
using two unsupervised objectives: Masked Language Modelling (MLM) and 
Next Sentence Prediction (NSP). In MLM, 15% of input tokens are randomly masked 
and the model must predict them from context. This bidirectional training allows 
BERT to condition on both left and right context simultaneously, unlike GPT which 
is strictly left-to-right. NSP trains the model to predict whether two sentences 
are consecutive in the original document. BERT-base has 12 transformer layers, 
768 hidden dimensions, and 12 attention heads, totalling 110 million parameters. 
BERT set new records on 11 NLP benchmarks upon release."""
    },
    {
        "id": "doc_003",
        "title": "Word Embeddings",
        "text": """Word embeddings map discrete vocabulary tokens to dense continuous vectors in 
a lower-dimensional space. Word2Vec, introduced by Mikolov et al. in 2013, 
learns embeddings by training a shallow neural network to predict a word from 
its context (CBOW) or predict context words from a target word (Skip-gram). 
The resulting vectors capture semantic relationships: the famous example 
vec(king) - vec(man) + vec(woman) ≈ vec(queen) demonstrates that analogical 
relationships are encoded geometrically. GloVe (Global Vectors) trains on 
global word co-occurrence statistics rather than local context windows. 
FastText extends Word2Vec by representing words as bags of character n-grams, 
allowing embeddings for out-of-vocabulary words. Unlike contextual embeddings 
from BERT, Word2Vec produces a single static vector per word."""
    },
    {
        "id": "doc_004",
        "title": "Named Entity Recognition",
        "text": """Named Entity Recognition (NER) is the task of identifying and classifying 
named entities in text into predefined categories such as person (PER), 
organisation (ORG), location (LOC), and miscellaneous (MISC). Traditional 
approaches used hand-crafted features with Conditional Random Fields (CRF) or 
Hidden Markov Models (HMM). Modern neural NER systems typically use a BiLSTM 
encoder followed by a CRF output layer, or fine-tuned transformer models like 
BERT. The standard evaluation metric is entity-level F1 score computed on the 
CoNLL-2003 benchmark. Common annotation schemes include BIO (Beginning, Inside, 
Outside) and BIOES (Beginning, Inside, Other, End, Single). Transfer learning 
from pre-trained language models has greatly reduced the need for large annotated 
NER corpora."""
    },
    {
        "id": "doc_005",
        "title": "Evaluation Metrics for NLP",
        "text": """Evaluating NLP systems requires metrics appropriate for each task. 
For text classification, accuracy, precision, recall, and F1 are standard. 
For language generation tasks, automatic metrics include BLEU (Bilingual 
Evaluation Understudy), which measures n-gram overlap between generated and 
reference texts; ROUGE (Recall-Oriented Understudy for Gisting Evaluation), 
used for summarisation; and METEOR, which accounts for stemming and synonyms. 
BERTScore uses contextual embeddings to measure semantic similarity rather than 
exact n-gram matches. Perplexity measures how well a language model predicts 
a held-out test set — lower perplexity indicates a better model. 
Human evaluation remains the gold standard for generation quality, 
but is costly and difficult to scale. Benchmark datasets like GLUE, SuperGLUE, 
and BIG-Bench allow systematic comparison across many tasks."""
    },
]

print(f"Knowledge base: {len(DOCUMENTS)} documents")
for doc in DOCUMENTS:
    print(f"  {doc['id']}: {doc['title']} ({len(doc['text'].split())} words)")

In [ ]:
# Coding Task A — Fixed-size chunking with overlap

def chunk_fixed_size(text: str, chunk_size: int = 150, overlap: int = 30) -> list:
    """
    Split text into overlapping fixed-size chunks measured in words.

    Args:
        text       : input text string
        chunk_size : number of words per chunk
        overlap    : number of words to overlap between consecutive chunks
    Returns:
        list of chunk strings
    """
    words = text.split()
    chunks = []
    step = chunk_size - overlap

    # TODO: iterate with step `step`, take slice words[i : i+chunk_size]
    # and join back into a string. Stop when start index >= len(words).
    i = 0
    while i < len(words):
        chunk = words[i : i + chunk_size]  # TODO: slice
        chunks.append(" ".join(chunk))
        i += step  # TODO: advance by step

    return chunks


# Coding Task B — Sentence-boundary chunking

def chunk_by_sentences(text: str, sentences_per_chunk: int = 3, overlap: int = 1) -> list:
    """
    Split text into chunks of `sentences_per_chunk` sentences with `overlap` sentence overlap.

    Args:
        text                : input text string
        sentences_per_chunk : number of sentences per chunk
        overlap             : number of sentences to overlap between chunks
    Returns:
        list of chunk strings
    """
    # Simple sentence splitter on '.', '!', '?' followed by whitespace or end
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks = []
    step = sentences_per_chunk - overlap
    # TODO: similar sliding window as above but over sentences
    i = 0
    while i < len(sentences):
        chunk_sents = sentences[i : i + sentences_per_chunk]  # TODO
        chunks.append(" ".join(chunk_sents))
        i += step

    return chunks


# Test both chunkers on the first document
doc = DOCUMENTS[0]["text"]
fixed_chunks = chunk_fixed_size(doc, chunk_size=80, overlap=20)
sent_chunks  = chunk_by_sentences(doc, sentences_per_chunk=2, overlap=1)

print(f"Document word count   : {len(doc.split())}")
print(f"Fixed-size chunks     : {len(fixed_chunks)}")
print(f"Sentence-based chunks : {len(sent_chunks)}")
print(f"\nFirst fixed chunk:\n{fixed_chunks[0]}")
print(f"\nFirst sentence chunk:\n{sent_chunks[0]}")

In [ ]:
# Coding Task C — Build a chunked corpus from all documents

def build_chunk_corpus(documents: list, chunk_size: int = 80, overlap: int = 20) -> list:
    """
    Chunk all documents and return a flat list of dicts:
      {'chunk_id': str, 'doc_id': str, 'title': str, 'text': str}
    chunk_id format: '{doc_id}_chunk_{i}'
    """
    corpus = []
    for doc in documents:
        chunks = chunk_fixed_size(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for i, chunk_text in enumerate(chunks):
            corpus.append({
                "chunk_id": f"{doc['id']}_chunk_{i}",
                "doc_id":   doc["id"],
                "title":    doc["title"],
                "text":     chunk_text,
            })
    return corpus


CHUNK_CORPUS = build_chunk_corpus(DOCUMENTS, chunk_size=80, overlap=20)

print(f"Total chunks: {len(CHUNK_CORPUS)}")
print("\nChunks per document:")
from collections import Counter
for doc_id, count in Counter(c["doc_id"] for c in CHUNK_CORPUS).items():
    print(f"  {doc_id}: {count} chunks")

print(f"\nSample chunk:\n{CHUNK_CORPUS[3]}")

### Questions
1. What effect does **increasing overlap** have on retrieval quality and storage cost?
2. Why might sentence-boundary chunking produce better retrieval results than fixed-size chunking for certain query types?
3. Describe a case where **very small chunks** would hurt retrieval and a case where **very large chunks** would hurt retrieval.

_Your answers here._

---
## Part 3 — Embeddings and Vector Store

Each chunk is mapped to a dense vector using an **embedding model**. We then store these vectors in a **vector store** and find the nearest neighbours to a query vector at retrieval time.

In this lab we use **TF-IDF** as a lightweight sparse embedding (no GPU required). The cosine similarity between TF-IDF vectors approximates lexical relevance. In production, dense embedding models (e.g. `sentence-transformers/all-MiniLM-L6-v2`) give much better semantic retrieval.

In [ ]:
# Coding Task A — Build a TF-IDF vector store

class TFIDFVectorStore:
    """
    A simple vector store backed by TF-IDF embeddings and cosine similarity.
    """

    def __init__(self):
        self.vectoriser = TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            stop_words="english",
        )
        self.chunks = []      # list of chunk dicts
        self.matrix = None    # sparse TF-IDF matrix, shape (n_chunks, vocab)

    def add_documents(self, chunks: list):
        """
        Embed all chunks and store them.

        Args:
            chunks: list of {'chunk_id', 'doc_id', 'title', 'text'} dicts
        """
        self.chunks = chunks
        texts = [c["text"] for c in chunks]
        # TODO: fit the vectoriser on `texts` and transform to get self.matrix
        self.matrix = self.vectoriser.fit_transform(texts)  # TODO

    def query(self, query_text: str, k: int = 3) -> list:
        """
        Retrieve the top-k most similar chunks to `query_text`.

        Returns:
            list of (chunk_dict, similarity_score) tuples, sorted by score desc
        """
        # TODO:
        # 1. transform query_text with self.vectoriser
        # 2. compute cosine similarity between query vector and self.matrix
        # 3. return top-k (chunk, score) pairs
        query_vec = self.vectoriser.transform([query_text])          # TODO
        sims = cosine_similarity(query_vec, self.matrix).flatten()   # TODO
        top_k_idx = np.argsort(sims)[::-1][:k]                      # TODO
        return [(self.chunks[i], float(sims[i])) for i in top_k_idx]


# Build the vector store
vs = TFIDFVectorStore()
vs.add_documents(CHUNK_CORPUS)
print(f"Vector store: {len(vs.chunks)} chunks, vocab size {vs.matrix.shape[1]}")

In [ ]:
# Coding Task B — Test retrieval on sample queries

SAMPLE_QUERIES = [
    "How does self-attention work in transformers?",
    "What training objectives does BERT use?",
    "How are word embeddings evaluated using analogies?",
    "What metrics are used to evaluate NER systems?",
    "What is BLEU score and when is it used?",
]

for query in SAMPLE_QUERIES:
    results = vs.query(query, k=3)
    print(f"\nQuery: {query}")
    print("-" * 60)
    for chunk, score in results:
        print(f"  [{score:.3f}] ({chunk['doc_id']}) {chunk['text'][:80]}...")

### Questions
1. Why is **cosine similarity** preferred over Euclidean distance for comparing TF-IDF vectors?
2. What is the main limitation of TF-IDF embeddings compared to dense sentence embeddings (e.g. from `sentence-transformers`)? Give a concrete example of a query that TF-IDF would handle poorly.
3. What is an **inverted index** and how does it speed up sparse vector retrieval?

_Your answers here._

---
## Part 4 — Retrieval Evaluation

To evaluate a retrieval system we need:
- A set of **queries**
- **Relevance judgements**: which documents are relevant to each query
- **Ranked retrieval results** from our system

Key metrics:
- **Precision@k**: fraction of the top-k retrieved items that are relevant
- **Recall@k**: fraction of all relevant items that appear in the top-k
- **Mean Reciprocal Rank (MRR)**: average of 1/rank of the first relevant result

$$\text{MRR} = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \frac{1}{\text{rank}_i}$$

In [ ]:
# Evaluation dataset: queries with ground-truth relevant doc_ids
EVAL_QUERIES = [
    {"query": "How does self-attention work?",
     "relevant_docs": {"doc_001"}},
    {"query": "What is masked language modelling?",
     "relevant_docs": {"doc_002"}},
    {"query": "Explain word2vec skip-gram training",
     "relevant_docs": {"doc_003"}},
    {"query": "BIO tagging scheme for entity recognition",
     "relevant_docs": {"doc_004"}},
    {"query": "How is BLEU score computed for machine translation?",
     "relevant_docs": {"doc_005"}},
    {"query": "Positional encodings in transformer models",
     "relevant_docs": {"doc_001"}},
    {"query": "BERT architecture number of layers and parameters",
     "relevant_docs": {"doc_002"}},
    {"query": "Out-of-vocabulary words in word embeddings",
     "relevant_docs": {"doc_003"}},
]

print(f"Evaluation set: {len(EVAL_QUERIES)} queries")

In [ ]:
# Coding Task A — Precision@k and Recall@k

def precision_at_k(retrieved_doc_ids: list, relevant_docs: set, k: int) -> float:
    """
    Compute precision@k: fraction of top-k retrieved docs that are relevant.

    Args:
        retrieved_doc_ids : ordered list of retrieved doc_ids (best first)
        relevant_docs     : set of ground-truth relevant doc_ids
        k                 : cutoff
    Returns:
        float in [0, 1]
    """
    top_k = retrieved_doc_ids[:k]
    # TODO: count how many of top_k are in relevant_docs, divide by k
    hits = sum(1 for d in top_k if d in relevant_docs)  # TODO
    return hits / k  # TODO


def recall_at_k(retrieved_doc_ids: list, relevant_docs: set, k: int) -> float:
    """
    Compute recall@k: fraction of relevant docs found in top-k.

    Args:
        retrieved_doc_ids : ordered list of retrieved doc_ids (best first)
        relevant_docs     : set of ground-truth relevant doc_ids
        k                 : cutoff
    Returns:
        float in [0, 1]
    """
    if not relevant_docs:
        return 0.0
    top_k = retrieved_doc_ids[:k]
    # TODO: count relevant docs found in top_k, divide by total relevant
    hits = sum(1 for d in top_k if d in relevant_docs)  # TODO
    return hits / len(relevant_docs)  # TODO


# Coding Task B — Mean Reciprocal Rank

def reciprocal_rank(retrieved_doc_ids: list, relevant_docs: set) -> float:
    """
    Compute the reciprocal rank: 1/rank of the first relevant result.
    Returns 0 if no relevant document is retrieved.
    """
    for rank, doc_id in enumerate(retrieved_doc_ids, start=1):
        if doc_id in relevant_docs:
            return 1.0 / rank  # TODO
    return 0.0  # TODO: no relevant doc found


def mean_reciprocal_rank(queries: list, retrieval_fn, k: int = 5) -> float:
    """
    Compute MRR over a list of query dicts.
    Each dict has 'query' and 'relevant_docs' keys.
    retrieval_fn(query, k) -> list of (chunk_dict, score)
    """
    rrs = []
    for item in queries:
        results = retrieval_fn(item["query"], k)
        retrieved_doc_ids = [r[0]["doc_id"] for r in results]
        rr = reciprocal_rank(retrieved_doc_ids, item["relevant_docs"])
        rrs.append(rr)
    # TODO: return the mean of rrs
    return sum(rrs) / len(rrs)  # TODO


# Evaluate
K = 3
prec_scores, rec_scores = [], []

print(f"{'Query':<45} {'P@{}':<8} {'R@{}':<8} {'RR'}".format(K, K))
print("-" * 72)

for item in EVAL_QUERIES:
    results = vs.query(item["query"], k=K)
    retrieved_ids = [r[0]["doc_id"] for r in results]
    p = precision_at_k(retrieved_ids, item["relevant_docs"], K)
    r = recall_at_k(retrieved_ids, item["relevant_docs"], K)
    rr = reciprocal_rank(retrieved_ids, item["relevant_docs"])
    prec_scores.append(p)
    rec_scores.append(r)
    print(f"{item['query'][:44]:<45} {p:<8.3f} {r:<8.3f} {rr:.3f}")

print("-" * 72)
print(f"{'MEAN':<45} {sum(prec_scores)/len(prec_scores):<8.3f} {sum(rec_scores)/len(rec_scores):<8.3f}", end="")
print(f" {mean_reciprocal_rank(EVAL_QUERIES, vs.query, K):.3f}")

In [ ]:
# Coding Task C — Plot P@k and R@k curves for k = 1..5

K_VALUES = list(range(1, 6))
mean_prec = []
mean_rec  = []

for k in K_VALUES:
    ps, rs = [], []
    for item in EVAL_QUERIES:
        results = vs.query(item["query"], k=k)
        retrieved_ids = [r[0]["doc_id"] for r in results]
        ps.append(precision_at_k(retrieved_ids, item["relevant_docs"], k))
        rs.append(recall_at_k(retrieved_ids, item["relevant_docs"], k))
    mean_prec.append(sum(ps) / len(ps))
    mean_rec.append(sum(rs) / len(rs))

fig, ax = plt.subplots(figsize=(7, 4))
# TODO: plot mean_prec vs K_VALUES (label='Precision@k')
# TODO: plot mean_rec  vs K_VALUES (label='Recall@k')
ax.plot(K_VALUES, mean_prec, marker='o', label='Precision@k')  # TODO
ax.plot(K_VALUES, mean_rec,  marker='s', label='Recall@k')    # TODO
ax.set_xlabel("k")
ax.set_ylabel("Score")
ax.set_title("Retrieval: Precision@k and Recall@k")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("retrieval_pk_rk.png", dpi=100)
plt.show()

### Questions
1. Explain the **precision-recall tradeoff** as k increases. Why do they move in opposite directions?
2. Why is MRR a useful metric for retrieval tasks where only the **first** relevant result matters?
3. Our evaluation treats each query as having a **single** relevant document. How would you handle multi-document relevance in a real evaluation?

_Your answers here._

---
## Part 5 — The RAG Pipeline

A RAG pipeline has three stages:
1. **Retrieve** — find the top-k most relevant chunks for a query.
2. **Augment** — insert the retrieved chunks into a prompt template.
3. **Generate** — pass the augmented prompt to a language model.

In this lab we simulate the generator with a simple extractive function (since we do not have access to a live LLM API). You will implement the retrieval and augmentation stages in full.

In [ ]:
# Coding Task A — Prompt template construction

PROMPT_TEMPLATE = """You are a helpful NLP teaching assistant. Use only the provided context 
to answer the question. If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {question}

Answer:"""


def build_rag_prompt(query: str, retrieved_chunks: list, max_context_words: int = 300) -> str:
    """
    Construct a RAG prompt by inserting retrieved chunks into the template.

    Args:
        query             : user question string
        retrieved_chunks  : list of (chunk_dict, score) tuples from the vector store
        max_context_words : truncate context if it exceeds this many words
    Returns:
        formatted prompt string
    """
    # TODO: concatenate the text of each retrieved chunk, numbered 1, 2, ...
    # Truncate the combined context to max_context_words words.
    context_parts = []
    word_count = 0
    for i, (chunk, score) in enumerate(retrieved_chunks, 1):
        part = f"[{i}] ({chunk['title']}) {chunk['text']}"
        words = part.split()
        if word_count + len(words) > max_context_words:
            words = words[: max_context_words - word_count]
            context_parts.append(" ".join(words))
            break
        context_parts.append(part)
        word_count += len(words)

    context = "\n\n".join(context_parts)  # TODO
    return PROMPT_TEMPLATE.format(context=context, question=query)


# Test prompt construction
query = "How does self-attention work in transformers?"
results = vs.query(query, k=3)
prompt = build_rag_prompt(query, results)
print(prompt)

In [ ]:
# Coding Task B — Simulated extractive generator
# (Replace this with a real LLM call when an API key is available)

def extractive_generate(prompt: str, max_answer_words: int = 50) -> str:
    """
    Simulated generator: extracts the sentence from the context that best
    matches the question using TF-IDF overlap. This is a stand-in for a
    real LLM. In production you would call an LLM API here.
    """
    # Split prompt at 'Context:' and 'Question:'
    parts = prompt.split("Context:\n", 1)
    if len(parts) < 2:
        return "[No context found.]"
    rest = parts[1]
    context_part, question_part = rest.split("\nQuestion:", 1)
    question = question_part.split("\nAnswer:")[0].strip()

    # Extract individual sentences from context
    sentences = re.split(r'(?<=[.!?])\s+', context_part)
    sentences = [s.strip() for s in sentences if len(s.split()) > 5]

    if not sentences:
        return "[Could not extract answer from context.]"

    # Score each sentence by TF-IDF cosine similarity with the question
    local_vec = TfidfVectorizer(stop_words="english")
    try:
        mat = local_vec.fit_transform(sentences + [question])
        sims = cosine_similarity(mat[-1:], mat[:-1]).flatten()
        best_idx = int(np.argmax(sims))
        answer = " ".join(sentences[best_idx].split()[:max_answer_words])
        return answer
    except ValueError:
        return sentences[0][:max_answer_words * 6]


def rag_answer(query: str, vector_store, k: int = 3) -> dict:
    """
    Full RAG pipeline: retrieve → augment → generate.

    Returns dict with keys: 'query', 'retrieved', 'prompt', 'answer'
    """
    # Step 1: Retrieve
    retrieved = vector_store.query(query, k=k)  # TODO

    # Step 2: Augment
    prompt = build_rag_prompt(query, retrieved)  # TODO

    # Step 3: Generate
    answer = extractive_generate(prompt)  # TODO

    return {
        "query":     query,
        "retrieved": retrieved,
        "prompt":    prompt,
        "answer":    answer,
    }


# Run RAG on sample queries
for query in SAMPLE_QUERIES:
    result = rag_answer(query, vs, k=3)
    print(f"Q: {result['query']}")
    print(f"A: {result['answer']}")
    print(f"   (top source: {result['retrieved'][0][0]['title']}, score={result['retrieved'][0][1]:.3f})")
    print()

### Optional Extension (if API key available)

Replace `extractive_generate` with a real LLM call. Fill in your API key and model below.

```python
import anthropic

client = anthropic.Anthropic(api_key="YOUR_API_KEY")

def llm_generate(prompt: str) -> str:
    message = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=256,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text
```

_Test your LLM-backed RAG pipeline on all SAMPLE_QUERIES and compare answers to the extractive baseline._

### Questions
1. What information does the **prompt template** communicate to the language model? Why is the instruction "Use only the provided context" important?
2. How would you modify the RAG pipeline to handle a query that requires information from **multiple documents**?
3. What is **re-ranking** in RAG and why is it useful as a second retrieval stage?

_Your answers here._

---
## Part 6 — Hallucination Analysis and Context Quality

When the retrieved context is irrelevant or missing the answer, the generator may **hallucinate** — confidently producing incorrect text. In this part you will study how retrieval quality affects generation fidelity.

In [ ]:
# Coding Task A — Out-of-scope queries (no relevant document exists)

OUT_OF_SCOPE = [
    "What is the current weather in Hanoi?",
    "Who won the FIFA World Cup 2026?",
    "What is the population of Vietnam?",
    "How do I train a diffusion model for image generation?",
]

print("=== Out-of-scope queries (no relevant document in KB) ===")
for query in OUT_OF_SCOPE:
    result = rag_answer(query, vs, k=3)
    top_score = result["retrieved"][0][1]
    print(f"\nQ: {query}")
    print(f"   Top retrieval score : {top_score:.3f}")
    print(f"   Top retrieved doc   : {result['retrieved'][0][0]['title']}")
    print(f"   Generated answer    : {result['answer'][:120]}...")

In [ ]:
# Coding Task B — Retrieval score threshold for abstention

def rag_with_threshold(query: str, vector_store, k: int = 3, threshold: float = 0.05) -> dict:
    """
    RAG pipeline with abstention: if the top retrieval score is below `threshold`,
    return a canned "I don't know" response instead of generating from weak context.

    Args:
        query        : user question
        vector_store : the TFIDFVectorStore
        k            : number of documents to retrieve
        threshold    : minimum cosine similarity to proceed with generation
    Returns:
        dict with keys 'query', 'answer', 'abstained', 'top_score'
    """
    retrieved = vector_store.query(query, k=k)
    top_score = retrieved[0][1] if retrieved else 0.0

    # TODO: if top_score < threshold, set answer to "I don't know" and abstained=True
    if top_score < threshold:
        return {"query": query, "answer": "I don't know — no relevant information found.",
                "abstained": True, "top_score": top_score}  # TODO

    prompt = build_rag_prompt(query, retrieved)
    answer = extractive_generate(prompt)
    return {"query": query, "answer": answer, "abstained": False, "top_score": top_score}


# Test threshold across in-scope and out-of-scope queries
ALL_TEST = SAMPLE_QUERIES + OUT_OF_SCOPE
THRESHOLD = 0.10

print(f"Threshold = {THRESHOLD}\n")
print(f"{'Query':<50} {'Score':>6} {'Abstained?'}")
print("-" * 70)
for q in ALL_TEST:
    r = rag_with_threshold(q, vs, threshold=THRESHOLD)
    print(f"{q[:49]:<50} {r['top_score']:>6.3f} {r['abstained']}")

In [ ]:
# Coding Task C — Effect of k on answer quality

# Measure extractive answer overlap (word overlap F1) with a reference sentence
REFERENCE_ANSWERS = {
    "How does self-attention work in transformers?": 
        "attention is computed as softmax QK^T sqrt d_k times V",
    "What training objectives does BERT use?":
        "Masked Language Modelling and Next Sentence Prediction",
    "How are word embeddings evaluated using analogies?":
        "king minus man plus woman equals queen demonstrates analogical relationships",
}

def word_overlap_f1(pred: str, ref: str) -> float:
    """Token-level F1 between prediction and reference (ignores case)."""
    pred_tokens = set(pred.lower().split())
    ref_tokens  = set(ref.lower().split())
    if not pred_tokens or not ref_tokens:
        return 0.0
    tp = len(pred_tokens & ref_tokens)
    precision = tp / len(pred_tokens)
    recall    = tp / len(ref_tokens)
    if precision + recall == 0:
        return 0.0
    # TODO: compute F1 = 2 * precision * recall / (precision + recall)
    return 2 * precision * recall / (precision + recall)  # TODO


print("Effect of k on answer word-overlap F1:")
print(f"{'Query':<45} " + " ".join(f"k={k:>4}" for k in range(1, 6)))
print("-" * 80)
for query, ref in REFERENCE_ANSWERS.items():
    f1s = []
    for k in range(1, 6):
        result = rag_answer(query, vs, k=k)
        f1s.append(word_overlap_f1(result["answer"], ref))
    print(f"{query[:44]:<45} " + " ".join(f"{f:>6.3f}" for f in f1s))

### Questions
1. What is **hallucination** in the context of LLMs? Distinguish between **intrinsic** hallucination (contradicts the context) and **extrinsic** hallucination (not verifiable from context).
2. The threshold approach in Task 6B avoids generating from poor context. What are the risks of setting the threshold **too high** vs **too low**?
3. How does **k** (number of retrieved chunks) affect the risk of hallucination? Is more context always better?
4. Describe two approaches other than retrieval-score thresholding that can be used to detect or reduce hallucination.

_Your answers here._

---
## Part 7 — Advanced RAG Concepts (Discussion)

### 7.1 Hybrid Retrieval

Modern RAG systems often combine **sparse retrieval** (BM25/TF-IDF) and **dense retrieval** (neural embeddings) via score fusion. The Reciprocal Rank Fusion (RRF) formula is:

$$\text{RRF}(d) = \sum_{r \in \text{rankers}} \frac{1}{k + r(d)}$$

where $r(d)$ is the rank of document $d$ in ranker $r$ and $k$ is a constant (typically 60).

**Task:** Given two ranked lists below, compute RRF scores for each document with k=60 and produce the merged ranking.

Sparse ranker: [doc_001, doc_003, doc_002, doc_004, doc_005]  
Dense ranker:  [doc_003, doc_001, doc_005, doc_002, doc_004]

_Show your calculation here._

---

### 7.2 Indexing Strategies

For each indexing strategy, describe one advantage and one disadvantage:

| Strategy | Advantage | Disadvantage |
|----------|-----------|------------|
| Flat exact search | | |
| IVF (Inverted File Index) | | |
| HNSW (Hierarchical Navigable Small World) | | |
| Product Quantisation (PQ) | | |

---

### 7.3 RAG vs Fine-Tuning Trade-offs

Complete the comparison table:

| Criterion | RAG | Fine-Tuning |
|-----------|-----|-------------|
| Up-to-date knowledge | | |
| Transparency / citability | | |
| Computational cost at inference | | |
| Handles domain-specific style | | |
| Risk of hallucination | | |
| Latency | | |

_Your answers here._

---
## Submission

- Complete all `TODO` sections in the code cells.
- Answer all written questions in the markdown cells above.
- Include in your submission:
  - The **P@k / R@k curve plot** from Part 4 (`retrieval_pk_rk.png`)
  - The **retrieval evaluation table** from Part 4 with mean P@3, R@3, and MRR
  - The **threshold abstention table** from Part 6B
  - The **k vs F1 table** from Part 6C
- Submit your completed `.ipynb` file as instructed.